# Spatial Correlation Analysis

This notebook demonstrates time-resolved spatial correlation analysis of a phasefield5d simulation run.

**Workflow:**
1. Compute correlations from raw `.npz` data with `make-correlations` (CLI) or `analyze_run()` (Python).
2. Load the saved results here for interactive visualisation.

**The visualisation parameters** (colour limits, selected time steps, component labels, axis ranges) need interactive tuning — that is why the visualisation lives here rather than in the CLI.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from phasefield5d.analysis.correlation import (
    analyze_run,
    save_results,
    load_results,
    plot_autocorr_map,
    plot_selected_pair_map,
    plot_zero_lag_matrix,
    plot_zero_lag_pair_traces,
    plot_autocorr_traces_at_selected_times,
    plot_domain_length_vs_time,
    plot_eigenvalue_map,
    plot_eigenvector_bar,
    plot_spectral_map,
    linestyles_from_pairs,
)

## 1  Point to your run

In [ ]:
# Edit this path to your run directory
RUN_DIR = Path("../results/Fe60Mn10Ni20Co5Cu5_at_873k_1dim/elastic_cubic_direction100/cells1600_dx1e-08_...")

RESULTS_FILE = RUN_DIR / "correlation_results.npz"

## 2  Compute or load

**Option A — CLI (recommended for large runs, run once):**
```bash
make-correlations results/my_run/
# or with options:
make-correlations results/my_run/ --max-lag 200 --kspace
```

**Option B — Python API (small runs or custom parameters):**

In [ ]:
if RESULTS_FILE.exists():
    print("Loading pre-computed results …")
    results = load_results(RESULTS_FILE)
else:
    print("Computing correlations (this may take a while for large runs) …")
    results = analyze_run(
        RUN_DIR,
        max_lag=None,       # None → N//4
        periodic=True,
        compute_kspace=True,
    )
    save_results(results, RESULTS_FILE)
    print(f"Saved → {RESULTS_FILE}")

# Inspect what we have
print("Keys:", list(results.keys()))
print("C_lij shape (n_time, n_lag, n_comp, n_comp):", results["C_lij"].shape)
print("Time range: {:.2e} … {:.2e} s".format(results["time"][0], results["time"][-1]))

## 3  Metadata

In [ ]:
elements = results["elements"]   # e.g. ['Fe','Mn','Ni','Co','Cu']
dx_nm    = results["dx"] * 1e9  # cell size in nm
time     = results["time"]       # shape (n_time,)
lag_nm   = results["lag_nm"]     # shape (n_lag,)

C_lij    = results["C_lij"]      # (n_time, n_lag, 5, 5)
C_lii    = results["C_lii"]      # (n_time, n_lag, 5)  autocorrelations

print("Elements:", elements)
print(f"dx = {dx_nm:.2f} nm, max lag = {lag_nm[-1]:.1f} nm")

## 4  Autocorrelation maps

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)
for i, ax in enumerate(axes):
    plot_autocorr_map(C_lii, time, lag_nm, component=i, ax=ax,
                      title=elements[i], vmax=None)
fig.suptitle("Autocorrelation C(lag, t) per element")
fig.tight_layout()
plt.show()

## 5  Zero-lag correlation matrix over time

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
plot_zero_lag_matrix(C_lij, time, elements=elements, ax=ax)
fig.tight_layout()
plt.show()

## 6  Autocorrelation traces at selected times

In [ ]:
# Pick a few time indices to compare
n_time = len(time)
selected_indices = [0, n_time // 4, n_time // 2, 3 * n_time // 4, n_time - 1]

fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)
for i, ax in enumerate(axes):
    plot_autocorr_traces_at_selected_times(
        C_lii, time, lag_nm,
        component=i,
        selected_indices=selected_indices,
        ax=ax,
        title=elements[i],
    )
fig.suptitle("Autocorrelation profiles at selected times")
fig.tight_layout()
plt.show()

## 7  Domain length vs. time

In [ ]:
features = results.get("features")  # dict of arrays, keyed by component index

if features is not None:
    fig, ax = plt.subplots(figsize=(7, 4))
    plot_domain_length_vs_time(features, time, elements=elements, ax=ax)
    ax.set_xscale("log")
    ax.set_yscale("log")
    fig.tight_layout()
    plt.show()
else:
    print("No feature data found. Re-run analyze_run() with feature_min_lag set.")

## 8  Eigenvalue decomposition

In [ ]:
eigvals = results.get("eigvals")  # (n_time, n_lag, 5)
eigvecs = results.get("eigvecs")  # (n_time, n_lag, 5, 5)

if eigvals is not None:
    fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)
    for mode in range(5):
        plot_eigenvalue_map(eigvals, time, lag_nm, mode=mode, ax=axes[mode],
                            title=f"mode {mode+1}")
    fig.suptitle("Eigenvalues of C_lij(t)")
    fig.tight_layout()
    plt.show()
else:
    print("Eigendecomposition not in results. Run analyze_run() with compute_eig=True.")

## 9  k-space spectral correlation (optional)

Only available when `--kspace` was passed to `make-correlations`.

In [ ]:
R_kij = results.get("R_kij")   # (n_time, n_k, 5, 5)
k     = results.get("k")        # (n_k,)  wavenumbers [1/nm]

if R_kij is not None:
    fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)
    for i, ax in enumerate(axes):
        plot_spectral_map(R_kij, time, k, component=i, ax=ax,
                          title=elements[i])
    fig.suptitle("Spectral autocorrelation R(k, t) per element")
    fig.tight_layout()
    plt.show()
else:
    print("k-space data not in results. Re-run with --kspace.")